## PDF Extraction and Embedding (Programme Documents)

### 1. Extracting raw text data

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import os, re

In [2]:
pdf_path = "../RAG/ProgramBooklet"

path = os.path.join(pdf_path)
pdfs = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
print(pdfs)

['MSc_EIE_46011_2526.pdf', 'BEngBSc_Scheme_IAIE_46409_2526.pdf', 'PhDMPhil_EEE_46601_2526.pdf', 'MSc_EE_46010_2526.pdf', 'BEng_Scheme_EE_46408_2526.pdf', 'MSc_EV_46012_2526.pdf', 'MSc_MQ_46013_2526.pdf']


In [3]:
def regex_enhance(txt):
    text = re.sub(r' {2,}', ' ', txt)  # Strip excess white spaces
    return text

docs = []
unwanted_metadata = ["producer", "creator", "creationdate", "file_path", 
                     "format", "title", "subject", "keywords", "moddate", 
                     "author", "trapped", "modDate", "creationDate"]

for pdf in pdfs:
    loader = PyMuPDFLoader(f"{pdf_path}/{pdf}")
    cur_pdf = loader.load()
    for doc in cur_pdf:
        # Augment metadata for PDFs
        doc.page_content = regex_enhance(doc.page_content)
        doc.metadata["source"] = pdf
        doc.metadata["content_type"] = "pdf"
        for key in unwanted_metadata:
            del doc.metadata[key]
    docs.extend(cur_pdf)

In [4]:
print(f"Extracted number of documents in PDFs: {len(docs)}")
'''
for doc in docs:
    print(f"Source: {doc.metadata.get('source')}, type: {doc.metadata.get('content_type')}")
'''
print(docs[2])

Extracted number of documents in PDFs: 1162
page_content='Master of Science in Electronic and Information Engineering 2025/26 
 
1 
 
1 
General Information 
 
1.1 
Programme Information 
 
Programme Title 
 
Master of Science in Electronic and Information Engineering 
電子及資訊工程學理學碩士學位 
 
Host Department 
 
Department of Electrical and Electronic Engineering (46011) 
 
Mode of Study and Normal Duration 
 
Mode 
Normal Duration 
Mixed-Mode 
Full-time: 1.5 years (3 semesters) 
Part-time: 2.5 years (5 semesters) 
 
Students should complete the programme within the normal duration of the programme. Those 
who exceed the normal duration of the programme will be de-registered from the programme 
unless prior approval has been obtained from relevant authorities. 
 
Award Title 
 
Students will be awarded one of the following awards upon successful completion of the 
required content of the respective award (specialism study options in brackets): 
 
• Master of Science in Electronic and Informat

### 2. Text Splitting

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    page = chunk.metadata.get("page", "N/A")
    chunk.metadata["chunk_id"] = f"PolyU_Doc_{source}_page_{page}_chunk_{i}"

In [6]:
print(chunks[50])

page_content='completion of the late assessment. 
 
The student concerned is required to submit his/her application for late assessment in writing to the 
Head of Department offering the subject, within five working days from the date of the examination, 
together with any original supporting documents. Approval of applications for late assessment 
and the means for such late assessments shall be given by the Head of Department offering the 
subject or the subject teacher concerned, in consultation with the Programme Leader. Verification 
of the supporting documents with the issuing authority may be conducted by the subject offering 
Department as part of the approval process. 
 
5.11 Assessment to be competed 
 
For cases where students fail marginally in one of the components within a subject, the BoE can 
defer making a decision until the students concerned have completed the necessary remedial work 
to the satisfaction of the subject examiner(s). The remedial work must not take the

### 3. Document Embedding in Chroma

In [7]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "polyu_eee_document" if not SINGLE else "vaa_documents"

In [8]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="./chroma_db")
collection = client.get_collection(name=collection_name)

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_26831/3659695745.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!


In [9]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i)]
    )

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

print(f"Added {len(chunks)} chunks into ChromaDB")

Added 3179 chunks into ChromaDB


/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_26831/2981975304.py:14: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


### 4. Simple Testing

In [10]:
query = "What is Master of Science of Electronic and Information Engineering?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content}...")
    print(f"Source: {result.metadata.get('source')}, Page: {result.metadata.get('page')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

Content: Master of Science in Electrical Engineering 2025/26 
 
1 
 
1 
General Information 
 
1.1 
Programme Information 
 
Programme Title (Code) 
 
Master of Science in Electrical Engineering (46010) 
電機工程學理學碩士學位 
 
Host Department 
 
Department of Electrical and Electronic Engineering 
 
Mode of Study and Normal Duration 
 
Mode 
Normal Duration 
Mixed-Mode 
Full-time: 1.5 years (3 semesters) 
Part-time: 2.5 years (5 semesters) 
 
Students should complete the programme within the normal duration of the programme. Those 
who exceed the normal duration of the programme will be de-registered from the programme 
unless prior approval has been obtained from relevant authorities. 
 
Award Title 
 
Students will be awarded one of the following awards upon successful completion of the 
required content of the respective award (specialism study options in brackets): 
 
 Master of Science in Electrical Engineering...
Source: MSc_EE_46010_2526.pdf, Page: 2
Chunk ID: PolyU_Doc_MSc_EE_46010_25